# NeoOLAF / RAGTree — DocRED raw-text entity + relation full run + evaluation

This notebook is for the **raw-text setting**: the model must extract **both entities and relations from the document text**.

Important distinction:

- Extraction prompt uses: raw text + title + global DocRED relation vocabulary.
- Extraction prompt does **not** use DocRED source/gold entity IDs.
- Evaluation uses gold entities only after prediction, to compute entity P/R/F1 and strict relation P/R/F1.

Outputs:

- entity inventory precision/recall/F1
- entity endpoint precision/recall/F1
- strict relation precision/recall/F1
- per-document CSV/JSON metrics

In [ ]:
from pathlib import Path
import os, json, subprocess, time, shlex

ROOT = Path("/home/galencarmedeiro/git/postdoc/NeoOLAF")
DATASET_JSONL = Path("/home/galencarmedeiro/git/postdoc/ragtree/data/preprocessed/docred_causal.jsonl")
RUNS_DIR = ROOT / "examples/RAGTreeDatasets/runs"
LOGS_DIR = RUNS_DIR / "logs"
LOGS_DIR.mkdir(parents=True, exist_ok=True)

TYPE_FILTER = "dev"
MAX_DOCS = 5             # smoke test first; set None for full selected split
DOCUMENT_WORKERS = 2    # smoke test first; use 8/12 for full run later

MODEL_NAME = "openai/gpt-oss-20b"
HOST = "https://openrouter.ai/api"
MAX_TOKENS = 8192
REQUEST_TIMEOUT = 600
TEMPERATURE = 0.0

# Raw-text extraction: no gold/source entity IDs are shown to the model.
DOCRED_RAW_RETRIES = 3
DOCRED_RAW_RETRY_SLEEP = 2.0
DOCRED_RAW_SCORING_CALIBRATION = True
DOCRED_RAW_DISABLE_HINTS = False
DOCRED_RAW_FOCUS_RELATION_IDS = None
DOCRED_RAW_MAX_RELATIONS = None
TEXT_CHAR_LIMIT = None

PRED_PATH = RUNS_DIR / "neoolaf_docred_raw_er_smoke5_predictions.canonical.jsonl"
SUMMARY_PATH = RUNS_DIR / "neoolaf_docred_raw_er_smoke5_run_summary.json"
ERROR_PATH = RUNS_DIR / "neoolaf_docred_raw_er_smoke5_errors.jsonl"
ARTIFACTS_ROOT = RUNS_DIR / "neoolaf_docred_raw_er_smoke5_artifacts"
VOCAB_PATH = RUNS_DIR / "docred_allowed_relations_raw_er_smoke5.json"
LOG_PATH = LOGS_DIR / "neoolaf_docred_raw_er_smoke5.log"
PID_PATH = RUNS_DIR / "neoolaf_docred_raw_er_smoke5.pid"
EVAL_JSON_PATH = RUNS_DIR / "neoolaf_docred_raw_er_smoke5_eval.json"
EVAL_CSV_PATH = RUNS_DIR / "neoolaf_docred_raw_er_smoke5_eval_per_doc.csv"

print("ROOT=", ROOT)
print("DATASET_JSONL=", DATASET_JSONL)
print("PRED_PATH=", PRED_PATH)
print("LOG_PATH=", LOG_PATH)
print("OPENROUTER_API_KEY present:", bool(os.environ.get("OPENROUTER_API_KEY")))

## 1. Start full raw-text entity + relation extraction

This launches the full dev split in the background. It writes JSONL predictions incrementally, so you can monitor progress.

In [ ]:
assert ROOT.exists(), ROOT
assert DATASET_JSONL.exists(), DATASET_JSONL
assert os.environ.get("OPENROUTER_API_KEY"), "Set OPENROUTER_API_KEY in the shell before starting Jupyter. Do not put the key in the notebook."

for p in [PRED_PATH, SUMMARY_PATH, ERROR_PATH, LOG_PATH]:
    p.unlink(missing_ok=True)
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)

cmd = [
    "python", str(ROOT / "experiments/methods/run_neoolaf_docred_raw_er.py"),
    "--dataset-jsonl-path", str(DATASET_JSONL),
    "--output-jsonl-path", str(PRED_PATH),
    "--summary-output-path", str(SUMMARY_PATH),
    "--error-log-jsonl-path", str(ERROR_PATH),
    "--artifacts-root", str(ARTIFACTS_ROOT),
    "--type-filter", TYPE_FILTER,
    "--document-workers", str(DOCUMENT_WORKERS),
    "--backend-name", "openrouter",
    "--host", HOST,
    "--model-name", MODEL_NAME,
    "--temperature", str(TEMPERATURE),
    "--max-tokens", str(MAX_TOKENS),
    "--request-timeout", str(REQUEST_TIMEOUT),
    "--openrouter-reasoning-effort", "minimal",
    "--openrouter-exclude-reasoning",
    "--relation-vocab-dataset-path", str(DATASET_JSONL),
    "--relation-vocab-output-path", str(VOCAB_PATH),
    "--docred-raw-retries", str(DOCRED_RAW_RETRIES),
    "--docred-raw-retry-sleep", str(DOCRED_RAW_RETRY_SLEEP),
]
if MAX_DOCS is not None:
    cmd.extend(["--max-docs", str(MAX_DOCS)])
if DOCRED_RAW_SCORING_CALIBRATION:
    cmd.append("--docred-raw-scoring-calibration")
if DOCRED_RAW_DISABLE_HINTS:
    cmd.append("--docred-raw-disable-hints")
if DOCRED_RAW_FOCUS_RELATION_IDS:
    cmd.extend(["--docred-raw-focus-relation-ids", DOCRED_RAW_FOCUS_RELATION_IDS])
if DOCRED_RAW_MAX_RELATIONS is not None:
    cmd.extend(["--docred-raw-max-relations", str(DOCRED_RAW_MAX_RELATIONS)])
if TEXT_CHAR_LIMIT is not None:
    cmd.extend(["--text-char-limit", str(TEXT_CHAR_LIMIT)])

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

log_f = open(LOG_PATH, "w", encoding="utf-8")
proc = subprocess.Popen(cmd, cwd=str(ROOT), stdout=log_f, stderr=subprocess.STDOUT, env=os.environ.copy())
PID_PATH.write_text(str(proc.pid), encoding="utf-8")
print("Started PID", proc.pid)
print("PID_PATH=", PID_PATH)

## 2. Monitor progress

In [ ]:
# Run this cell repeatedly.
import os, json

pred_lines = 0
if PRED_PATH.exists():
    pred_lines = sum(1 for _ in PRED_PATH.open(encoding="utf-8"))
print("prediction lines:", pred_lines)

if SUMMARY_PATH.exists():
    print("summary:")
    print(SUMMARY_PATH.read_text(encoding="utf-8"))

if ERROR_PATH.exists():
    print("errors:", sum(1 for _ in ERROR_PATH.open(encoding="utf-8")))

if LOG_PATH.exists():
    print("last log lines:")
    lines = LOG_PATH.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(lines[-30:]))

## 3. Evaluate entity and relation extraction

This computes:

- **entity_inventory**: predicted raw entities mapped to DocRED gold entity clusters by exact alias matching.
- **entity_endpoint**: gold entities that appear as endpoints in relation triples.
- **relation_strict**: strict relation triples after mapping predicted entity labels/local IDs to gold entity IDs. This is the relation score that matters most for DocRED-style triples.

In [ ]:
eval_cmd = [
    "python", str(ROOT / "experiments/methods/evaluate_docred_raw_er.py"),
    "--gold-jsonl-path", str(DATASET_JSONL),
    "--pred-jsonl-path", str(PRED_PATH),
    "--type-filter", TYPE_FILTER,
    "--output-json-path", str(EVAL_JSON_PATH),
    "--output-csv-path", str(EVAL_CSV_PATH),
]
print(" ".join(shlex.quote(x) for x in eval_cmd))
res = subprocess.run(eval_cmd, cwd=str(ROOT), text=True, capture_output=True)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError("evaluation failed")
print("EVAL_JSON_PATH=", EVAL_JSON_PATH)
print("EVAL_CSV_PATH=", EVAL_CSV_PATH)

In [ ]:
# Pretty-print global metrics.
metrics = json.loads(EVAL_JSON_PATH.read_text(encoding="utf-8"))
for key in ["entity_inventory", "entity_endpoint", "relation_strict"]:
    m = metrics[key]
    print("\n" + key)
    print(f"gold={m['gold']} pred={m['pred']} TP={m['TP']} FP={m['FP']} FN={m['FN']}")
    print(f"precision={m['precision']:.4f} recall={m['recall']:.4f} f1={m['f1']:.4f}")

## 4. Inspect a few predictions

In [ ]:
for i, line in enumerate(PRED_PATH.open(encoding="utf-8")):
    if i >= 5:
        break
    r = json.loads(line)
    print("\n" + "=" * 100)
    print(r.get("document_id"), "|", r.get("title"), "ok=", r.get("parsed_ok"))
    ents = r.get("prediction", {}).get("entities", [])
    rels = r.get("prediction", {}).get("relations", [])
    print("entities:", len(ents), "relations:", len(rels))
    for e in ents[:15]:
        print(f"  ENT {e.get('entity_id')}: {e.get('label')} [{e.get('type')}]")
    for rel in rels[:20]:
        print(f"  REL {rel.get('head')} -- {rel.get('relation_id')} --> {rel.get('tail')}")

## 5. Package results

In [ ]:
zip_path = ROOT / "neoolaf_docred_raw_er_smoke5_results.zip"
files = [
    PRED_PATH,
    SUMMARY_PATH,
    ERROR_PATH,
    VOCAB_PATH,
    EVAL_JSON_PATH,
    EVAL_CSV_PATH,
    LOG_PATH,
    ROOT / "experiments/methods/run_neoolaf_docred_raw_er.py",
    ROOT / "experiments/methods/evaluate_docred_raw_er.py",
    ROOT / "examples/RAGTreeDatasets/RAGTreeDatasets_DocRED_Raw_EntityRelation_Full_Run_Eval_NeoOLAF.ipynb",
]
existing = [str(p.relative_to(ROOT)) for p in files if p.exists()]
cmd = ["zip", "-r", str(zip_path), *existing]
print(" ".join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, cwd=str(ROOT), check=True)
print(zip_path)